# ATM 407: can convection keep up?

You are the scientist responsible for one atmospheric column inside a global model. The dynamical core will try to lift and destabilize that column; the physics packages will try to restore a balanced state. Your job is to predict which process wins, implement the dynamical forcing, and defend your interpretation with diagnostics.

### Learning goals

By the end of the lab you should be able to:

1. connect hydrostatic balance to atmospheric mass on a sigma grid;
2. diagnose static stability using potential temperature and $N^2$;
3. distinguish resolved dynamical forcing from parameterized physics tendencies;
4. implement vertical advection and adiabatic temperature change in pressure coordinates; and
5. explain how forcing and convective-adjustment timescales control CAPE and rainfall.

The SCM has no horizontal pressure-gradient force, Coriolis acceleration, or internally resolved circulation. Here, large-scale vertical motion is supplied externally, exactly as an observationally forced SCM or a host dynamical core would supply it.

**Lab rule:** make a written prediction before every experiment. A wrong forecast with a good physical explanation is more valuable than a correct guess made afterward.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/evanwellmeyer/GCM/blob/main/notebooks/02_experiments_atm407.ipynb)

## Colab setup

Run this cell first. It installs the current model and verifies that Python can import it.

In [ ]:
from pathlib import Path
import json
import subprocess
import sys

incolab = 'google.colab' in sys.modules
if incolab:
    root = Path('/content/GCM')
    if not root.exists():
        subprocess.run([
            'git', 'clone', '--depth', '1',
            'https://github.com/evanwellmeyer/GCM.git', str(root),
        ], check=True)
else:
    root = Path.cwd().resolve()
    while root != root.parent and not (root / 'pyproject.toml').exists():
        root = root.parent

if not (root / 'pyproject.toml').exists():
    raise FileNotFoundError('Open this notebook from inside the GCM repository.')
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

from importlib import metadata, util

try:
    metadata.version('gcm-scm')
    installed = util.find_spec('matplotlib') is not None
except metadata.PackageNotFoundError:
    installed = False

if not installed:
    subprocess.run([
        sys.executable, '-m', 'pip', 'install', '--quiet', '-e', f'{root}[plot]',
    ], check=True)
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

import scm
print('SCM ready from', Path(scm.__file__).resolve())

In [ ]:
from copy import deepcopy
import time

import matplotlib.pyplot as plt
import numpy as np
import torch

from scm.column_model import initial_state, physics_step, run, update_derived
from scm.configuration import extract_param_overrides, load_run_config
from scm.ensemble import default_params
from scm.thermo import Rd, cp, g, geopotential, make_grid, relative_humidity

torch.manual_seed(0)
device = torch.device('cpu')
print('device:', device)

## Meet your atmospheric column

We begin from a saved, nearly steady radiative-convective column rather than spending class time on a long spin-up. Radiation, surface fluxes, and convection are already active, so this is a moving balance--not a motionless atmosphere.

The reference has small energy imbalances, moderate CAPE, precipitation dominated by deep convection, and no active numerical limiters. It is an idealized laboratory atmosphere, not an observed climatological sounding. Think of it as the control member against which every forecast will be judged.

The code cell below contains plumbing used throughout the lab. You do not need to memorize it. The governing equations that you are responsible for will be derived and implemented later in the lesson.


In [ ]:
experiment = {
    'nlevels': 20,
    'dt': 900.0,
    'days': 3,
    'diagnostic_hours': 3,
    'radiation_steps': 8,
    'surface_temperature': 290.0,
    'surface_pressure': 100000.0,
    'solar_constant': 1360.0,
    'zenith_factor': 0.25,
    'ocean_depth': 50.0,
    'surface_albedo': 0.32,
    'wind_speed': 5.0,
}

referencemetadata = json.loads(
    (root / 'notebooks/data/atm407_equilibrium_20level.json').read_text()
)
print('reference configuration:', referencemetadata['configuration_label'])
print(f"reference surface temperature: {referencemetadata['surface_temperature_k']:.2f} K")
print(f"reference CAPE: {referencemetadata['cape_jkg']:.0f} J kg-1")
print(f"reference precipitation: {referencemetadata['precipitation_mmday']:.2f} mm day-1")
print(f"mass at or above 95% RH: {referencemetadata['rh95_mass_fraction']:.0%}")
print(f"mass-flux cap-active fraction: {referencemetadata['mass_flux_cap_fraction']:.0%}")

def makeparams(settings, updates=None):
    params = default_params(device=device)
    params.update(extract_param_overrides(load_run_config()))
    params.update({
        'dt': settings['dt'],
        'ps0': settings['surface_pressure'],
        'ts_init': settings['surface_temperature'],
        'solar_constant': settings['solar_constant'],
        'zenith_factor': settings['zenith_factor'],
        'ocean_depth': settings['ocean_depth'],
        'albedo': settings['surface_albedo'],
        'wind_speed': settings['wind_speed'],
        'convection_scheme': 'mass_flux',
        'radiation_scheme': 'multiband',
        'use_slab_ocean': True,
        'profile_diagnostics': True,
    })
    if updates is not None:
        params.update(updates)
    return params

def loadreference(nlevels=20, batch=1):
    reference = np.load(root / 'notebooks/data/atm407_equilibrium_20level.npz')
    settings = dict(experiment)
    settings['nlevels'] = nlevels
    grid = make_grid(nlevels, device=device)
    params = makeparams(settings)
    state = initial_state(batch, grid, params, device=device)
    sourcesigma = reference['sigma_full']
    targetsigma = grid['sigma_full'].cpu().numpy()

    for name in ['t', 'q', 'qc', 'cloud_fraction']:
        profile = np.interp(targetsigma, sourcesigma, reference[name])
        values = torch.as_tensor(profile, dtype=state[name].dtype, device=device)
        state[name] = values.unsqueeze(0).repeat(batch, 1)

    referencegrid = make_grid(len(sourcesigma), device=device)
    sourcedsigma = referencegrid['dsigma'].cpu().numpy()
    sourcewater = np.sum(reference['q'] * sourcedsigma)
    targetwater = torch.sum(state['q'][0] * grid['dsigma']).item()
    state['q'] = state['q'] * (sourcewater / targetwater)
    state['ts'].fill_(float(reference['ts']))
    state['ps'].fill_(float(reference['ps']))
    state['slab_ts_ref'] = state['ts'].clone()
    state['slab_energy'].zero_()
    return update_derived(state, grid)

def integrate(settings, updates=None, batch=1, state=None, lsforcing=None):
    grid = make_grid(settings['nlevels'], device=device)
    params = makeparams(settings, updates)
    if state is None:
        state = initial_state(batch, grid, params, device=device)
    stepsperday = round(86400 / settings['dt'])
    nsteps = round(settings['days'] * stepsperday)
    diagnosticsteps = max(1, round(settings['diagnostic_hours'] * 3600 / settings['dt']))
    start = time.perf_counter()
    state, history = run(
        state, grid, params, nsteps,
        rad_interval=settings['radiation_steps'],
        diag_interval=diagnosticsteps,
        ls_forcing=lsforcing,
    )
    elapsed = time.perf_counter() - start
    return grid, params, state, history, elapsed

def series(history, name, member=0, scale=1.0):
    values = [entry[name][member].detach().cpu().item() for entry in history]
    return np.array(values) * scale

def days(history, dt):
    return np.array([entry['step'] for entry in history]) * dt / 86400

## Mission 1: weigh the atmosphere without a scale

Hydrostatic balance says $dp=-\rho g\,dz$. Integrating through one layer and dividing by area gives

$$m_{layer}/A=\frac{\Delta p}{g}.$$

That is why a pressure thickness is also a mass coordinate. This model uses $\sigma=p/p_s$, so its levels move with surface pressure while retaining a fixed fractional position in the column.

**Forecast before plotting:** Are the layers equally spaced in pressure? Which portion of the atmosphere contains the most mass?

> **Your prediction:** Write one or two sentences here before running the cell.

Run the calculation, verify that $\sum\Delta p/g=p_s/g$, and explain why this identity is a useful conservation check for a dynamical core.

In [ ]:
grid = make_grid(experiment['nlevels'], device=device)
params = makeparams(experiment)
state = loadreference(experiment['nlevels'])
pressure = state['p'][0].cpu().numpy() / 100
deltap = state['dp'][0].cpu().numpy() / 100
levels = np.arange(experiment['nlevels'])

fig, axes = plt.subplots(1, 2, figsize=(9, 5), sharey=True)
axes[0].plot(pressure, levels, marker='o')
axes[0].set_xlabel('full-level pressure (hPa)')
axes[1].barh(levels, deltap)
axes[1].set_xlabel('layer pressure thickness (hPa)')
axes[0].set_ylabel('model level')
axes[0].invert_yaxis()
for ax in axes:
    ax.grid(alpha=0.3)
fig.tight_layout()
plt.show()

massfromlayers = state['dp'].sum().item() / g
massfromsurface = state['ps'].item() / g
print(f'layer sum: {massfromlayers:.2f} kg m-2')
print(f'ps / g:    {massfromsurface:.2f} kg m-2')

## Mission 2: find the atmosphere's weak spots

A rising dry parcel cools by expansion, so temperature alone does not tell us whether it will return to its starting level. Potential temperature removes that adiabatic pressure effect:

$$\theta=T\left(\frac{p_0}{p}\right)^{R_d/c_p}.$$

The dry Brunt--Vaisala frequency measures the restoring force:

$$N^2=\frac{g}{\theta}\frac{\partial\theta}{\partial z}.$$

If $N^2>0$, a displaced parcel oscillates; if $N^2<0$, dry overturning is favored. Small $N^2$ marks a layer that is easy to disturb.

**Forecast before plotting:** Where do you expect the strongest stability: boundary layer, free troposphere, or stratosphere?

> **Your prediction:** Record the layer and your physical reason.

Use the four panels to identify stable and unstable layers. A few negative values near the surface are not automatically a model failure: this is a discretized, turbulent boundary layer, while $N^2$ here is a dry parcel diagnostic. Treat them as a clue to discuss what the boundary-layer physics must continually mix. Then quantify the atmospheric mass at or above 95% RH and explain why this idealized column is not an observed tropical mean sounding.


In [ ]:
temperature = state['t'][0]
pressurepa = state['p'][0]
theta = temperature * (100000.0 / pressurepa) ** (Rd / cp)
rh = relative_humidity(state['q'], state['t'], state['p'])[0] * 100
height = geopotential(state['t'], state['q'], state['p'], grid)[0]
dthetadz = np.gradient(theta.cpu().numpy(), height.cpu().numpy())
n2 = g / theta.cpu().numpy() * dthetadz

fig, axes = plt.subplots(1, 4, figsize=(14, 5), sharey=True)
axes[0].plot(temperature.cpu(), pressure)
axes[0].set_xlabel('temperature (K)')
axes[1].plot(theta.cpu(), pressure)
axes[1].set_xlabel('potential temperature (K)')
axes[2].plot(rh.cpu(), pressure)
axes[2].plot(rh[rh >= 95].cpu(), pressure[rh.cpu().numpy() >= 95], 'o', color='tab:red')
axes[2].axvline(95, color='tab:red', linestyle='--', linewidth=0.8)
axes[2].set_xlabel('relative humidity (%)')
axes[3].plot(n2 * 1e4, pressure)
axes[3].axvline(0, color='black', linewidth=0.8)
axes[3].set_xlabel(r'$N^2$ ($10^{-4}$ s$^{-2}$)')
axes[0].set_ylabel('pressure (hPa)')
axes[0].invert_yaxis()
for ax in axes:
    ax.grid(alpha=0.3)
fig.tight_layout()
plt.show()
saturatedmass = state['dp'][0, rh >= 95].sum() / state['dp'][0].sum()
print(f'mass at or above 95% RH: {saturatedmass.item():.0%}')
print(f'height range: {height.min().item() / 1000:.1f} to {height.max().item() / 1000:.1f} km')
print(f'minimum dry N2: {n2.min():+.2e} s-2')
print('levels with dry N2 below zero (hPa):', np.round(pressure[n2 < 0], 1))

## Mission 3: become a tendency detective

A GCM usually separates its work into two pieces:

- the **dynamical core** transports mass, momentum, heat, and water;
- the **physics column** represents radiation, surface exchange, turbulence, convection, condensation, and clouds.

Radiation and surface exchange can change the column's total moist energy because they cross its boundaries. Mixing and convection mostly rearrange heat and water internally, so a column integral can hide large opposing tendencies at individual levels.

**Forecast before plotting:** Which process should cool the upper atmosphere? Which should warm and moisten the lowest levels? Which should transport heat upward?

> **Your prediction:** Assign at least three processes to an expected sign and layer.

After running the cell, choose one surprising tendency and explain its sign in physical terms.

In [ ]:
stepstate = deepcopy(state)
stepstate, diagnostics, radiationcache = physics_step(stepstate, grid, params)
boundarylabels = ['radiation', 'surface exchange']
boundaryvalues = [
    diagnostics['rad_energy_tendency'][0].item(),
    diagnostics['surface_energy_tendency'][0].item(),
]
processes = [
    ('radiation', 'radiation'),
    ('surface', 'surface'),
    ('boundary layer', 'boundary_layer'),
    ('shallow', 'shallow'),
    ('deep convection', 'deep'),
    ('condensation', 'condensation'),
    ('clouds', 'cloud'),
]

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
colors = ['tab:red' if value > 0 else 'tab:blue' for value in boundaryvalues]
axes[0].bar(boundarylabels, boundaryvalues, color=colors)
axes[0].axhline(0, color='black', linewidth=0.8)
axes[0].set_ylabel('atmospheric energy tendency (W m-2)')
axes[0].set_title('boundary contributions')
axes[0].tick_params(axis='x', rotation=20)
axes[0].grid(axis='y', alpha=0.3)

for label, name in processes:
    temperaturetendency = diagnostics[f'{name}_temperature_tendency'][0] * 86400
    moisturetendency = diagnostics[f'{name}_moisture_tendency'][0] * 86400 * 1000
    axes[1].plot(temperaturetendency.cpu(), pressure, label=label)
    axes[2].plot(moisturetendency.cpu(), pressure, label=label)
axes[1].set(xlabel='temperature tendency (K day-1)', ylabel='pressure (hPa)')
axes[2].set(xlabel='moisture tendency (g kg-1 day-1)', ylabel='pressure (hPa)')
for ax in axes[1:]:
    ax.axvline(0, color='black', linewidth=0.8)
    ax.invert_yaxis()
    ax.grid(alpha=0.3)
axes[2].legend(fontsize=8, loc='best')
fig.tight_layout()
plt.show()
print(f"TOA net flux: {diagnostics['toa_net'][0].item():+.2f} W m-2")
print(f"surface total flux: {diagnostics['surface_total_flux'][0].item():+.2f} W m-2")
print(f"column residual: {diagnostics['column_energy_residual'][0].item():+.2f} W m-2")

## Mission 4: build the dynamical forcing

Now you will write the bridge between dynamics and physics. In pressure coordinates, $\omega=Dp/Dt$ is negative for ascent. Ignoring horizontal advection,

$$\frac{\partial T}{\partial t}=\underbrace{-\omega\frac{\partial T}{\partial p}}_{\text{vertical advection}}+\underbrace{\frac{R_d}{c_p}\frac{T\omega}{p}}_{\text{adiabatic expansion}},$$

$$\frac{\partial q}{\partial t}=-\omega\frac{\partial q}{\partial p}.$$

The first temperature term moves the environmental profile through the column. The second is compressional heating or expansional cooling. Their sum determines the local temperature response.

We prescribe $\omega(p)$ to be zero at the top and surface and strongest near $\sigma=0.5$. It acts for one day; the column then adjusts freely for two days.

**Forecast before coding:** For ascent, predict the signs of vertical temperature advection, adiabatic expansion, total temperature tendency, and moisture tendency. Will CAPE and deep rain increase or decrease?

> **Your prediction:** Make a sign table and forecast the CAPE/rain response.

In [ ]:
# First implement the pressure derivatives used in the equations above.
def pressuregradient(field, pressure):
    gradient = torch.zeros_like(field)
    gradient[:, 1:-1] = (field[:, 2:] - field[:, :-2]) / (pressure[:, 2:] - pressure[:, :-2])
    gradient[:, 0] = (field[:, 1] - field[:, 0]) / (pressure[:, 1] - pressure[:, 0])
    gradient[:, -1] = (field[:, -1] - field[:, -2]) / (pressure[:, -1] - pressure[:, -2])
    return gradient

# This function recomputes the forcing from the evolving model state.
def ascentforcing(grid, peakomega, durationdays=1.0):
    peakomega = torch.as_tensor(peakomega, dtype=torch.float32, device=device).reshape(-1, 1)
    sigma = grid['sigma_full'].to(device=device, dtype=torch.float32).reshape(1, -1)
    shape = torch.sin(torch.pi * sigma).clamp(min=0.0)

    def forcing(step, state):
        if step * experiment['dt'] >= durationdays * 86400:
            return None
        pressure = state['p']
        omega = -peakomega.to(pressure.dtype) * shape.to(pressure.dtype)
        dtdp = pressuregradient(state['t'], pressure)
        dqdp = pressuregradient(state['q'], pressure)

        verticaladvection = -omega * dtdp
        adiabaticexpansion = (Rd / cp) * state['t'] * omega / pressure
        temperaturetendency = verticaladvection + adiabaticexpansion
        moisturetendency = -omega * dqdp
        return {
            'dt': temperaturetendency,
            'dq': moisturetendency,
            'verticaladvection': verticaladvection,
            'adiabaticexpansion': adiabaticexpansion,
        }

    return forcing

grid = make_grid(experiment['nlevels'], device=device)
peakomega = 0.05
forcing = ascentforcing(grid, [peakomega], durationdays=1.0)
previewstate = loadreference(experiment['nlevels'])
preview = forcing(0, previewstate)
omega = -peakomega * np.sin(np.pi * grid['sigma_full'].cpu().numpy())

fig, axes = plt.subplots(1, 4, figsize=(15, 5), sharey=True)
axes[0].plot(omega * 36, pressure)
axes[0].set(xlabel=r'$\omega$ (hPa hour$^{-1}$)', ylabel='pressure (hPa)')
axes[1].plot(preview['verticaladvection'][0].cpu() * 86400, pressure, label='vertical advection')
axes[1].plot(preview['adiabaticexpansion'][0].cpu() * 86400, pressure, label='adiabatic')
axes[1].set_xlabel('temperature terms (K day-1)')
axes[1].legend(fontsize=8)
axes[2].plot(preview['dt'][0].cpu() * 86400, pressure)
axes[2].set_xlabel('total temperature tendency (K day-1)')
axes[3].plot(preview['dq'][0].cpu() * 86400 * 1000, pressure)
axes[3].set_xlabel('moisture tendency (g kg-1 day-1)')
for ax in axes:
    ax.axvline(0, color='black', linewidth=0.8)
    ax.invert_yaxis()
    ax.grid(alpha=0.3)
fig.tight_layout()
plt.show()

controlstate = loadreference(experiment['nlevels'])
grid, params, controlstate, controlhistory, controlelapsed = integrate(
    experiment, state=controlstate
)
forcedstate = loadreference(experiment['nlevels'])
grid, params, forcedstate, forcedhistory, forcedelapsed = integrate(
    experiment, state=forcedstate, lsforcing=forcing
)
timeaxis = days(controlhistory, experiment['dt'])

fig, axes = plt.subplots(3, 1, figsize=(8, 8), sharex=True)
axes[0].plot(timeaxis, series(controlhistory, 'cape'), label='control')
axes[0].plot(timeaxis, series(forcedhistory, 'cape'), label='one day of ascent')
axes[0].set_ylabel('CAPE (J kg-1)')
axes[0].legend()
axes[1].plot(timeaxis, series(controlhistory, 'precip_conv', scale=86400), label='control deep rain')
axes[1].plot(timeaxis, series(forcedhistory, 'precip_conv', scale=86400), label='forced deep rain')
axes[1].set_ylabel('deep rain (mm day-1)')
axes[1].legend()
axes[2].plot(timeaxis, series(forcedhistory, 'forcing_energy_tendency'))
axes[2].axhline(0, color='black', linewidth=0.8)
axes[2].set(xlabel='model day', ylabel='imposed energy tendency (W m-2)')
for ax in axes:
    ax.grid(alpha=0.3)
fig.tight_layout()
plt.show()
print(f'maximum ascent: {-peakomega * 36:.2f} hPa hour-1')
print(f"control mean deep rain: {series(controlhistory, 'precip_conv', scale=86400).mean():.2f} mm day-1")
print(f"forced mean deep rain: {series(forcedhistory, 'precip_conv', scale=86400).mean():.2f} mm day-1")
print(f'total runtime: {controlelapsed + forcedelapsed:.1f} s')

## Final challenge: can convection keep up?

You now control two competing clocks:

- **dynamical forcing:** stronger ascent destabilizes and moistens the column faster;
- **convective adjustment:** a shorter CAPE-removal timescale lets parameterized convection respond faster.

Run nine columns spanning three ascent strengths and three convective timescales. Ascent acts for one day and each case runs for two days. Predict the corner with the largest CAPE buildup and the corner with the strongest deep rainfall.

**Lock in your forecast before revealing the heatmaps.**

> Largest CAPE buildup: ______ ascent and ______ convective response.
>
> Strongest deep rain: ______ ascent and ______ convective response.

All nine columns run simultaneously. Check that the mass-flux cap remains inactive; otherwise the experiment would test the limiter rather than the intended timescale competition.


In [ ]:
omegavalues = [0.02, 0.05, 0.10]
timescalevalues = [43200.0, 86400.0, 172800.0]
cases = [(omega, timescale) for omega in omegavalues for timescale in timescalevalues]
updates = {
    'tau_cape': torch.tensor([case[1] for case in cases], device=device),
}
challengesettings = dict(experiment)
challengesettings['days'] = 2
challengegrid = make_grid(experiment['nlevels'], device=device)
challengestart = loadreference(experiment['nlevels'], batch=len(cases))
challengeforcing = ascentforcing(
    challengegrid, [case[0] for case in cases], durationdays=1.0
)
grid, params, challengestate, challengehistory, elapsed = integrate(
    challengesettings, updates=updates, batch=len(cases), state=challengestart,
    lsforcing=challengeforcing,
)
capeanomalies = []
rainrates = []
capfractions = []
referencecape = referencemetadata['cape_jkg']
for member, case in enumerate(cases):
    cape = series(challengehistory, 'cape', member=member)
    capeanomalies.append(cape.max() - referencecape)
    rainrates.append(series(challengehistory, 'precip_conv', member=member, scale=86400).mean())
    capfractions.append(series(challengehistory, 'mass_flux_cap_active', member=member).mean())

capegrid = np.array(capeanomalies).reshape(len(omegavalues), len(timescalevalues))
raingrid = np.array(rainrates).reshape(len(omegavalues), len(timescalevalues))
fig, axes = plt.subplots(1, 2, figsize=(12, 5), constrained_layout=True)
for ax, values, title, label, colormap in [
    (axes[0], capegrid, 'maximum CAPE anomaly', 'CAPE anomaly (J kg-1)', 'coolwarm'),
    (axes[1], raingrid, 'mean deep precipitation', 'rain (mm day-1)', 'viridis'),
]:
    image = ax.imshow(values, origin='lower', aspect='auto', cmap=colormap)
    ax.set_xticks(range(len(timescalevalues)), [f'{value / 3600:.0f}' for value in timescalevalues])
    ax.set_yticks(range(len(omegavalues)), [f'{value * 36:.2f}' for value in omegavalues])
    ax.set(xlabel='convective timescale (hours)', ylabel='peak ascent magnitude (hPa hour-1)', title=title)
    fig.colorbar(image, ax=ax, label=label)
plt.show()

print(f'maximum cap-active fraction: {max(capfractions):.0%}')
winner = int(np.argmax(capeanomalies))
print('largest CAPE accumulation (omega, timescale):', cases[winner])
print(f'maximum CAPE anomaly: {capeanomalies[winner]:.1f} J kg-1')
print(f'batched challenge runtime: {elapsed:.1f} s')

## Boss level (optional): does the answer survive a new grid?

A result is more convincing when it does not depend strongly on an arbitrary grid choice. Interpolate the 20-level reference to 10, 20, and 40 levels, then give each grid one day to adjust. Compare CAPE, precipitation, and runtime.

This is a **remapping stress test**, not formal convergence: only the 20-level atmosphere was spun up on its native grid. Explain why CAPE can change when the same sounding is sampled differently, and design the separate native-grid integrations required for a real convergence claim.

In [ ]:
resolutionresults = []
for nlevels in [10, 20, 40]:
    settings = dict(experiment)
    settings['nlevels'] = nlevels
    settings['days'] = 1
    initialstate = loadreference(nlevels)
    diagnosticstate = deepcopy(initialstate)
    diagnosticgrid = make_grid(nlevels, device=device)
    diagnosticparams = makeparams(settings)
    diagnosticstate, initialdiagnostics, cache = physics_step(
        diagnosticstate, diagnosticgrid, diagnosticparams
    )
    grid, params, finalstate, history, elapsed = integrate(
        settings, state=initialstate
    )
    resolutionresults.append({
        'levels': nlevels,
        'initialcape': initialdiagnostics['cape'][0].item(),
        'cape': series(history, 'cape')[-1],
        'rain': series(history, 'precip_total', scale=86400).mean(),
        'runtime': elapsed,
    })

for result in resolutionresults:
    print(
        f"{result['levels']:2d} levels | "
        f"CAPE {result['initialcape']:7.1f} -> {result['cape']:7.1f} J kg-1 | "
        f"mean rain {result['rain']:5.2f} mm day-1 | "
        f"runtime {result['runtime']:4.1f} s"
    )

## Your forecast briefing

Submit the completed notebook and a short forecast briefing organized around four claims:

1. **Mass and stability:** Where is the column most stable, least stable, and most massive?
2. **Process diagnosis:** Which physics tendencies maintain the reference sounding?
3. **Forecast verification:** Did your predicted signs and challenge winner match the model? Explain any miss using the equations, not hindsight.
4. **Model limits:** What important atmospheric behavior cannot occur in a single column without a dynamical core?

Include one figure that you consider the strongest evidence for your interpretation. The numerical-sensitivity boss level is optional unless assigned by your instructor.